# Creation of a networkX graph based on Train Planning Rules (TPR)

## Import libraries and data

In [4]:
import networkx as nx
import pandas as pd

import xml.etree.ElementTree as ET
import csv

## Inputs

In [5]:
#Change inputs here

Train_Planning_file = "data/TrainPlanningRules.xml"
Stops = "data/Stops.csv"

## Transformation of the file into a csv file

In [6]:
nodes_file = "nodes.csv"
edges_file = "edges.csv"

### Extract nodes

In [7]:
with open(nodes_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "tiploc", "name"])

    for event, elem in ET.iterparse(Train_Planning_file, events=("end",)):
        if elem.tag == "TTPRLocation":
            
            writer.writerow([
                elem.attrib.get("Id"),
                elem.attrib.get("Tiploc"),
                elem.attrib.get("Description")
            ])
            
            elem.clear()

### Extract edges

In [8]:
with open(edges_file, "w", newline="", encoding="utf-8") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["line_code", "source", "target", "distance_miles"])

    for event, elem in ET.iterparse(Train_Planning_file, events=("end",)):
        if elem.tag == "TTPRLineOfRoute":
            line_code = elem.attrib.get("Abbreviation")
            last_location = None
            last_mileage = None


            for loc_elem in elem.findall(".//TTPRLorLocation"):
                current_location_raw = loc_elem.find("Location").attrib.get("FK")
                #Separation of id and name
                current_location = current_location_raw.split(":")[-1] if ":" in current_location_raw else current_location_raw
                
                try:
                    current_mileage = float(loc_elem.attrib.get("Mileage", 0))
                except ValueError:
                    current_mileage = 0.0

                #If there a previous node, we create the edge
                if last_location is not None:
                    distance = round(abs(current_mileage - last_mileage), 3)
                    writer.writerow([line_code, last_location, current_location, distance])

                last_location = current_location
                last_mileage = current_mileage
            
            elem.clear()

In [9]:
nodes = pd.read_csv(nodes_file)
edges = pd.read_csv(edges_file)

print(f"Nodes number : {len(nodes)}")
print(f"Edges number : {len(edges)}")
print(f"Connected nodes number : {len(set(edges['source']).union(set(edges['target'])))}")

Nodes number : 52713
Edges number : 7488
Connected nodes number : 6764


### Data cleaning

In [10]:
df_nodes = nodes.copy()
df_edges = edges.copy()

#Delete row without ID
df_nodes = df_nodes.dropna(subset=['id'])
df_nodes = df_nodes.drop_duplicates()

#Clean the str and make sure any blank space is there
df_nodes['tiploc'] = df_nodes['tiploc'].astype(str).str.strip()
df_nodes['name'] = df_nodes['name'].astype(str).str.strip()

invalid_values = ["unknown", "none", "nan"]

df_nodes = df_nodes[
    ~df_nodes.drop(columns=["id"])
      .apply(lambda col: col.astype(str).str.strip().str.lower())
      .isin(invalid_values)
      .all(axis=1)
]


df_nodes.to_csv("nodes_clean.csv", index=False)

print("Number of nodes remaining : ", len(df_nodes))

Number of nodes remaining :  12107


In [11]:
df_nodes.head()

,id,tiploc,name
2,3.0,AACHEN,Aachen
3,4.0,ABCWM,Abercwmboi
4,5.0,ABDAPEN,Penywaun
5,6.0,ABDARE,Aberdare
6,7.0,ABDATRE,Trecynon


## Graph creation

In [12]:
df_nodes['id'] = pd.to_numeric(df_nodes['id'], errors='coerce').fillna(0).astype(int).astype(str)
df_edges['source'] = pd.to_numeric(df_edges['source'], errors='coerce').fillna(0).astype(int).astype(str)
df_edges['target'] = pd.to_numeric(df_edges['target'], errors='coerce').fillna(0).astype(int).astype(str)

#Graph creation
G = nx.from_pandas_edgelist(df_edges, source='source', target='target', edge_attr='distance_miles')
nodes_with_edges = G.number_of_nodes()

#Quick diagnostic
print("===== Network diagnostic ===== \n")
print(f"Number of edges : {G.number_of_edges()}")
print(f"Number of connected nodes : {G.number_of_nodes()}")

#Connectivity checks

if nx.is_connected(G):
    print("Network is fully connected.")
else:
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    giant_size = len(components[0])
    print(f"Network fragmented in {len(components)} parts.")
    print(f"Giant Component : {giant_size} nodes")
    print(f"Coverage ratio : {(giant_size / nodes_with_edges)*100:.1f}%")

===== Network diagnostic ===== 

Number of edges : 7289
Number of connected nodes : 6764
Network fragmented in 7 parts.
Giant Component : 6751 nodes
Coverage ratio : 99.8%


## New graph removing unconnected parts

In [13]:
#We keep the Giant Component and will work only with this part, as the other components are not numerous and are not relevant for the analysis

main_component_nodes = max(nx.connected_components(G), key=len)
G_connected = G.subgraph(main_component_nodes).copy()

#Connectivity checks

if nx.is_connected(G_connected):
    print("Network is fully connected.")
else:
    components = sorted(nx.connected_components(G_connected), key=len, reverse=True)
    giant_size = len(components[0])
    print(f"Network fragmented in {len(components)} parts.")
    print(f"Giant Component : {giant_size} nodes")
    print(f"Coverage ratio : {(giant_size / nodes_with_edges)*100:.1f}%")

#Statistics on the new graph
print(f"Number of nodes updated : {G_connected.number_of_nodes()}")
print(f"Number of edges updated : {G_connected.number_of_edges()}")

Network is fully connected.
Number of nodes updated : 6751
Number of edges updated : 7282


## Identification of the nodes most linked with edges

In [14]:
#Top 13 most connected nodes (with edges)

print("\n ===== Hubs (Centrality) ===== \n")
degrees = dict(G_connected.degree())
top_hubs = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:13]

for node, deg in top_hubs:

    row = df_nodes[df_nodes['id'] == node]
    if not row.empty:
        name = row['name'].values[0]
        print(f"- {name} (ID: {node}) : {deg} connections")
    else:
        print(f"- ID {node} : {deg} connections (Name not found)")


 ===== Hubs (Centrality) ===== 

- Ashford International (ASHFKY) (ID: 360) : 8 connections
- Harlesden Jn (ID: 4120) : 7 connections
- Finsbury Park (ID: 3490) : 6 connections
- Longhedge Jn (ID: 5649) : 6 connections
- Nuneaton (ID: 6624) : 6 connections
- Acton Wells Jn (ID: 102) : 6 connections
- Carlisle South Junction (ID: 11001) : 6 connections
- Regional Boundary (ID: 10984) : 6 connections
- Ely North Jn (ID: 3195) : 6 connections
- Factory Jn (ID: 3337) : 6 connections
- Dundee Central Jn (ID: 2966) : 6 connections
- Portobello Jn (Edinburgh) (ID: 7278) : 6 connections
- Tulse Hill (ID: 9502) : 6 connections


## Identification of nodes with the highest betweenness centrality

In [15]:
print("\n ===== Nodes with a high betweenness centrality ===== \n")

centrality_dict = nx.betweenness_centrality(G_connected)
top_strategic = sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:10]

for node, score in top_strategic:
    row = df_nodes[df_nodes['id'] == node]
    if not row.empty:
        name = row['name'].values[0]
        print(f"- {name} (ID: {node}) : importance = {score:.2%}")
    else:
        print(f"- ID {node} : importance = {score:.2%}")


 ===== Nodes with a high betweenness centrality) ===== 

- Searchlight Lane Jn (ID: 10809) : importance = 39.87%
- Willesden No7 (ID: 10151) : importance = 36.10%
- Willesden Jn (Willesden No7) (ID: 11504) : importance = 34.20%
- Stone (ID: 8861) : importance = 33.84%
- Rugeley Trent Valley (ID: 7904) : importance = 33.79%
- Rugeley South Jn (ID: 7917) : importance = 33.60%
- Nuneaton Sig RN5433 (ID: 6632) : importance = 33.60%
- Up and Down Goods (ID: 12077) : importance = 33.59%
- Wolverton Signal KR1498 (ID: 11948) : importance = 33.59%
- Wolverton Signal KR1495 (ID: 11947) : importance = 33.59%


In [17]:
df_naptan = pd.read_csv(Stops, low_memory=False)
df_nodes_clean = pd.read_csv("nodes_clean.csv")

df_rail = df_naptan[df_naptan['StopType'] == 'RLY'].copy()
df_rail['tiploc_match'] = df_rail['ATCOCode'].str[4:]
df_geo_unique = df_rail.drop_duplicates(subset=['tiploc_match'])[['tiploc_match', 'Longitude', 'Latitude']]


df_nodes_geo = df_nodes_clean.merge(df_geo_unique, left_on='tiploc', right_on='tiploc_match', how='left')

print(f"Number of nodes : {len(df_nodes_geo)}")
print(f"Number of nodes with location : {df_nodes_geo['Latitude'].notna().sum()}")

Number of nodes : 12107
Number of nodes with location : 2661


In [27]:
df_nodes_clean.head(20)


,id,tiploc,name
0,3.0,AACHEN,Aachen
1,4.0,ABCWM,Abercwmboi
2,5.0,ABDAPEN,Penywaun
3,6.0,ABDARE,Aberdare
4,7.0,ABDATRE,Trecynon
5,8.0,ABDO,Aberdour
6,9.0,ABDVY,Aberdovey
7,10.0,ABER,Aber
8,11.0,ABGLELE,Abergele & Pensarn
9,12.0,ABGNWYN,Abergynolwyn


In [28]:
df_geo_unique.head(20)

,tiploc_match,Longitude,Latitude
428150,ABDARE,-3.443083,51.715058
428151,ASHYDN,-2.576700,51.478750
428152,SWNACFC,-2.055100,50.638300
428153,BEULYPK,0.518336,51.758057
428154,ELINTN,-2.657900,55.985600
428155,HBOLTN,-2.885591,53.491136
428156,THANETP,1.361777,51.330843
428157,MSBTN,-3.521330,50.704300
428158,REDGGPK,-1.001389,51.426667
428159,IVRNAIR,-4.055200,57.533500


## Map to show the graph

In [20]:
import folium

# Centrer la carte sur la moyenne des coordonnées
center_lat = df_nodes_geo["Latitude"].mean()
center_lon = df_nodes_geo["Longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

for _, row in df_nodes_geo.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=3,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        popup=f"ID: {row['id']}<br>Name: {row['name']}"
    ).add_to(m)

# Pour accélérer les accès
coords = df_nodes_geo.set_index("id")[["Latitude", "Longitude"]]

for _, row in df_edges.iterrows():
    src = coords.loc[row["source"]]
    tgt = coords.loc[row["target"]]

    folium.PolyLine(
        locations=[[src["Latitude"], src["Longitude"]], [tgt["Latitude"], tgt["Longitude"]]],
        color="red",
        weight=2,
        opacity=0.8
    ).add_to(m)

    m


ValueError: Location values cannot contain NaNs.

In [21]:
df_nodes_geo[df_nodes_geo["Latitude"].isna() | df_nodes_geo["Longitude"].isna()]


,id,tiploc,name,tiploc_match,Longitude,Latitude
0,3.0,AACHEN,Aachen,NaN,NaN,NaN
1,4.0,ABCWM,Abercwmboi,NaN,NaN,NaN
2,5.0,ABDAPEN,Penywaun,NaN,NaN,NaN
4,7.0,ABDATRE,Trecynon,NaN,NaN,NaN
9,12.0,ABGNWYN,Abergynolwyn,NaN,NaN,NaN
...,...,...,...,...,...,...
12102,12273.0,GLNTNJN,Glinton Jn,NaN,NaN,NaN
12103,12274.0,NaN,Seaham Engineering Siding,NaN,NaN,NaN
12104,12275.0,NaN,Wembley Stabling Siding,NaN,NaN,NaN
12105,12276.0,SESA17,Southease Signal TLW17,NaN,NaN,NaN


In [22]:
len(df_nodes_geo)

12107

In [24]:
df_naptan.columns


Index(['ATCOCode', 'NaptanCode', 'PlateCode', 'CleardownCode', 'CommonName',
       'CommonNameLang', 'ShortCommonName', 'ShortCommonNameLang', 'Landmark',
       'LandmarkLang', 'Street', 'StreetLang', 'Crossing', 'CrossingLang',
       'Indicator', 'IndicatorLang', 'Bearing', 'NptgLocalityCode',
       'LocalityName', 'ParentLocalityName', 'GrandParentLocalityName', 'Town',
       'TownLang', 'Suburb', 'SuburbLang', 'LocalityCentre', 'GridType',
       'Easting', 'Northing', 'Longitude', 'Latitude', 'StopType',
       'BusStopType', 'TimingStatus', 'DefaultWaitTime', 'Notes', 'NotesLang',
       'AdministrativeAreaCode', 'CreationDateTime', 'ModificationDateTime',
       'RevisionNumber', 'Modification', 'Status'],
      dtype='object')